<a href="https://colab.research.google.com/github/daehyun99/BurnFit-AI-developer-assignment/blob/main/data/01_02_training_generate_eval_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma3 모델 학습 및 평가용 데이터 생성
- Gemma3 모델을 지도학습하고, 평가용 데이터를 생성합니다.
- 코드를 실행하려면
  - `step` 값 수정 필요
  - google drive 경로 수정 필요

## 1. pip install

In [ ]:
!pip install -q gemma

In [ ]:
!pip install -q openpyxl

## 2. 세션 재시작 및 모델로드

In [ ]:
# prompt: google drive mount

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Common imports
import os
import jax
import jax.numpy as jnp
import optax
import treescope

# Gemma imports
from kauldron import kd
from gemma import gm
from gemma import peft

import pandas as pd

In [ ]:
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

In [ ]:
model = gm.nn.LoRA(
    rank=4,
    model=gm.nn.Gemma3_1B(tokens="batch.input"),
)

In [ ]:
token_ids = jnp.zeros((1, 256,), dtype=jnp.int32)  # Create the (batch_size, seq_length)

params = model.init(
    jax.random.key(0),
    token_ids,
)

params = params['params']

In [ ]:
treescope.show(params)

{
  'embedder': {'input_embedding': <jax.Array bfloat16(262144, 1152) ≈-0.00012 ±0.01 [≥-0.029, ≤0.025] nonzero:301_989_888>},
  'final_norm': {'scale': <jax.Array bfloat16(1152,) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:1_152>},
  'layer_0': {'attn': {'_key_norm': {'scale': <jax.Array bfloat16(256,) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:256>}, '_query_norm': {'scale': <jax.Array bfloat16(256,) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:256>}, 'attn_vec_einsum': {'_LoRAEinsum_0': {'lora': {'a': <jax.Array bfloat16(4, 256, 4) ≈-0.00064 ±0.044 [≥-0.077, ≤0.076] zero:23 nonzero:4_073>, 'b': <jax.Array bfloat16(4, 1152) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:4_608>}}, 'w': <jax.Array bfloat16(4, 256, 1152) ≈-0.00011 ±0.01 [≥-0.029, ≤0.025] nonzero:1_179_648>}, 'kv_einsum': {'_LoRAEinsum_0': {'lora': {'a': <jax.Array bfloat16(1152, 4) ≈-0.00022 ±0.042 [≥-0.072, ≤0.071] zero:35 nonzero:4_573>, 'b': <jax.Array bfloat16(4, 2, 1, 256) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:2_048>}}, 'w': <jax.Array bfloat16(2, 1, 1152, 256) ≈-0.00012 ±0.01 [≥-0.029, ≤0.025] nonzero:589_824>}, 'q_einsum': {'_LoRAEinsum_0': {'lora': {'a': <jax.Array bfloat16(1152, 4) ≈-0.00057 ±0.041 [≥-0.072, ≤0.071] zero:37 nonzero:4_571>, 'b': <jax.Array bfloat16(4, 4, 256) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:4_096>}}, 'w': <jax.Array bfloat16(4, 1152, 256) ≈-0.00013 ±0.01 [≥-0.029, ≤0.025] nonzero:1_179_648>}}, 'mlp': {'_LoRAEinsum_gating_einsum': {'lora': {'a': <jax.Array bfloat16(1152, 4) ≈-0.0016 ±0.041 [≥-0.072, ≤0.071] zero:42 nonzero:4_566>, 'b': <jax.Array bfloat16(4, 2, 6912) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:55_296>}}, '_LoRAEinsum_linear': {'lora': {'a': <jax.Array bfloat16(6912, 4) ≈-0.0002 ±0.017 [≥-0.03, ≤0.029] zero:211 nonzero:27_437>, 'b': <jax.Array bfloat16(4, 1152) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:4_608>}}, 'gating_einsum': <jax.Array bfloat16(2, 6912, 1152) ≈-0.00013 ±0.01 [≥-0.029, ≤0.025] nonzero:15_925_248>, 'linear': <jax.Array bfloat16(6912, 1152) ≈-0.00012 ±0.01 [≥-0.029, ≤0.025] nonzero:7_962_624>}, 'post_attention_norm': {'scale': <jax.Array bfloat16(1152,) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:1_152>}, 'post_ffw_norm': {'scale': <jax.Array bfloat16(1152,) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:1_152>}, 'pre_attention_norm': {'scale': <jax.Array bfloat16(1152,) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:1_152>}, 'pre_ffw_norm': {'scale': <jax.Array bfloat16(1152,) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:1_152>}},
  'layer_1': {'attn': {'_key_norm': {'scale': <jax.Array bfloat16(256,) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:256>}, '_query_norm': {'scale': <jax.Array bfloat16(256,) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:256>}, 'attn_vec_einsum': {'_LoRAEinsum_0': {'lora': {'a': <jax.Array bfloat16(4, 256, 4) ≈-0.00052 ±0.044 [≥-0.077, ≤0.076] zero:28 nonzero:4_068>, 'b': <jax.Array bfloat16(4, 1152) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:4_608>}}, 'w': <jax.Array bfloat16(4, 256, 1152) ≈-0.00014 ±0.01 [≥-0.029, ≤0.025] nonzero:1_179_648>}, 'kv_einsum': {'_LoRAEinsum_0': {'lora': {'a': <jax.Array bfloat16(1152, 4) ≈2.6e-06 ±0.042 [≥-0.072, ≤0.071] zero:31 nonzero:4_577>, 'b': <jax.Array bfloat16(4, 2, 1, 256) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:2_048>}}, 'w': <jax.Array bfloat16(2, 1, 1152, 256) ≈-0.00013 ±0.01 [≥-0.029, ≤0.025] nonzero:589_824>}, 'q_einsum': {'_LoRAEinsum_0': {'lora': {'a': <jax.Array bfloat16(1152, 4) ≈-0.00086 ±0.042 [≥-0.072, ≤0.071] zero:32 nonzero:4_576>, 'b': <jax.Array bfloat16(4, 4, 256) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:4_096>}}, 'w': <jax.Array bfloat16(4, 1152, 256) ≈-0.00014 ±0.01 [≥-0.029, ≤0.025] nonzero:1_179_648>}}, 'mlp': {'_LoRAEinsum_gating_einsum': {'lora': {'a': <jax.Array bfloat16(1152, 4) ≈-0.0008 ±0.042 [≥-0.072, ≤0.071] zero:39 nonzero:4_569>, 'b': <jax.Array bfloat16(4, 2, 6912) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:55_296>}}, '_LoRAEinsum_linear': {'lora': {'a': <jax.Array bfloat16(6912, 4) ≈-0.00039 ±0.017 [≥-0.03, ≤0.029] zero:227 nonzero:27_421>, 'b': <jax.Array bfloat16(4, 1152) ≈0.0 ±0.0 [≥0.0, ≤0.0] zero:4_608>}}, 'gating_einsum': <jax.Array bfloat16(2, 6912, 1152) ≈-0.00012 ±0.01 [≥-0.029, ≤0.025] nonzero:15_925_248>, 'linear': <jax.Array bfloat16(6912, 1152) ≈-0.00012 ±0.

# 반드시 step 값 확인 필요!

In [ ]:
step = 6 # 값 확인!

In [ ]:
class MyInitTransform:
    def transform(self, state):
        original = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)
        lora = gm.ckpts.load_params(f"/content/drive/MyDrive/Gemma3-lora{step-1}")
        merged = peft.merge_params(original, lora)
        return state.replace(params=merged)  # 중요! state에 params를 설정해야 함


In [ ]:
# # # 첫 로드인 경우에만
# init_transform = gm.ckpts.SkipLoRA(
#     wrapped=gm.ckpts.LoadCheckpoint(
#         path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
#     ),
# )

In [ ]:
optimizer = kd.optim.partial_updates(
    optax.adafactor(learning_rate=0.005),
    # We only optimize the LoRA weights. The rest of the model is frozen.
    mask=kd.optim.select("lora"),
)

In [ ]:
tokenizer = gm.text.Gemma3Tokenizer()

tokenizer.encode('This is an example sentence', add_bos=True)

[<_Gemma3SpecialTokens.BOS: 2>, 2094, 563, 614, 2591, 13315]

## 3. 학습 데이터 로드

In [ ]:
# data를 json = {question, answer 구조로 제작해서 json 형태로 만들기}
ds = kd.data.py.Json(
    f'/content/drive/MyDrive/05_[공유파일]/Burnfit_dataset/gemma3_formatted_for_training{step}.json',
    shuffle=False,
    batch_size=8,
    transforms=[
        gm.data.Seq2SeqTask(
            in_prompt='question',
            in_response='answer',
            out_input='input',
            out_target='target',
            out_target_mask='loss_mask',
            tokenizer=tokenizer,
            max_length=1024,
        ),
    ],
)

ex = ds[0]

treescope.show(ex)

{
  'input': # np.ndarray int64(8, 1024) [≥0, ≤247_339] zero:6_280 nonzero:1_912
    array([[   2,  105, 2364, ...,    0,    0,    0],
           [   2,  105, 2364, ...,    0,    0,    0],
           [   2,  105, 2364, ...,    0,    0,    0],
           ...,
           [   2,  105, 2364, ...,    0,    0,    0],
           [   2,  105, 2364, ...,    0,    0,    0],
           [   2,  105, 2364, ...,    0,    0,    0]])
  ,
  'loss_mask': <np.ndarray bool(8, 1024, 1) true:1_608 false:6_584>,
  'target': <np.ndarray int64(8, 1024, 1) [≥0, ≤247_339] zero:6_280 nonzero:1_912>,
}

In [ ]:
text = tokenizer.decode(ex['input'][0])

print(text)

<start_of_turn>user
Jim Wendler의 531 프로그램의 주요 구성 요소와 훈련 방법에 대해 자세히 설명해줄 수 있나요?<end_of_turn>
<start_of_turn>model
Jim Wendler의 531 프로그램은 근력 훈련을 위한 효과적인 루틴으로, 4주 주기로 구성되어 있습니다. 이 프로그램은 스쿼트, 벤치 프레스, 데드리프트, 오버헤드 프레스를 주요 리프트로 하며, 각 리프트에 대해 5회, 3회, 1회의 반복을 통해 점진적으로 중량을 증가시키는 방식입니다. 531의 핵심은 매주 중량을 조금씩 늘려가며, 각 주마다 다른 반복 수를 통해 근육의 적응을 유도하는 것입니다. 이 프로그램은 개인의 목표에 맞춰 조정할 수 있는 유연성을 제공하며, 근력 향상뿐만 아니라 운동의 재미를 더할 수 있도록 설계되었습니다.


## 4. 모델 학습

In [ ]:
trainer = kd.train.Trainer(
    seed=42,  # The seed of enlightenment
    workdir='/tmp/ckpts',  # TODO(epot): Make the workdir optional by default
    # Dataset
    train_ds=ds,
    # Model
    model=model,
    init_transform=MyInitTransform(), # init_transform=init_transform : 첫 번째인 경우만 # init_transform=MyInitTransform()
    # Training parameters
    num_train_steps=300,
    train_losses={
        "loss": kd.losses.SoftmaxCrossEntropyWithIntLabels(
            logits="preds.logits",
            labels="batch.target",
            mask="batch.loss_mask",
        ),
    },
    optimizer=optimizer,
)

In [ ]:
state, aux = trainer.train()

Disabling pygrain multi-processing (unsupported in colab).
Starting training loop at step 0


train:   0%|          | 0/301 [00:00<?, ?it/s]

In [ ]:
sampler = gm.text.ChatSampler(
    model=model,
    params=state.params,
    tokenizer=tokenizer,
)

## 5. 모델 파라미터 저장

In [ ]:
gm.ckpts.save_params(state.params, f'/content/drive/MyDrive/Gemma3-lora{step}')

## 6. Gemma3 추론 및 teacher 모델로 답변 평가를 위한 데이터프레임 제작

In [ ]:
# # 첫번째 실행
# df = pd.read_excel(f"/content/drive/MyDrive/05_[공유파일]/Burnfit_dataset/generated_data(gpt-4o-mini).xlsx")
# user_inputs = df["inputs"]

df = pd.read_excel(f"/content/drive/MyDrive/05_[공유파일]/Burnfit_dataset/eval_result{step-1}.xlsx")
user_inputs = df["inputs"]

In [ ]:
print(user_inputs[:5])

0    Jim Wendler의 531 프로그램의 주요 구성 요소와 훈련 방법에 대해 자세히...
1    5/3/1 운동 루틴의 구성 요소와 주간 스케줄에 대해 구체적으로 설명해 주실 수 ...
2      헬스 입문자로서 531 프로그램의 기본 개념과 구조를 간단히 설명해 주실 수 있나요?
3          531 루틴의 구성 요소와 훈련 방식을 좀 더 구체적으로 설명해줄 수 있나요?
4    Jim Wendler의 5/3/1 프로그램에서 각 주차별 세트와 반복 구성에 대한 ...
Name: inputs, dtype: object


In [ ]:
gemma3_outputs = []
index = 0
for user_input in user_inputs:
  output = sampler.chat(f'{user_input}')
  gemma3_outputs.append(output)
  print(f"index : {index} | ✅ Good")
  index += 1

df = pd.DataFrame()
df['user_input'] = user_inputs
df['gemma3_outputs'] = gemma3_outputs
df.head()

index : 0 | ✅ Good
index : 1 | ✅ Good
index : 2 | ✅ Good
index : 3 | ✅ Good
index : 4 | ✅ Good
index : 5 | ✅ Good
index : 6 | ✅ Good
index : 7 | ✅ Good
index : 8 | ✅ Good
index : 9 | ✅ Good
index : 10 | ✅ Good
index : 11 | ✅ Good
index : 12 | ✅ Good
index : 13 | ✅ Good
index : 14 | ✅ Good
index : 15 | ✅ Good
index : 16 | ✅ Good
index : 17 | ✅ Good
index : 18 | ✅ Good
index : 19 | ✅ Good
index : 20 | ✅ Good
index : 21 | ✅ Good
index : 22 | ✅ Good
index : 23 | ✅ Good
index : 24 | ✅ Good
index : 25 | ✅ Good
index : 26 | ✅ Good
index : 27 | ✅ Good
index : 28 | ✅ Good
index : 29 | ✅ Good


,user_input,gemma3_outputs
0,Jim Wendler의 531 프로그램의 주요 구성 요소와 훈련 방법에 대해 자세히...,"Jim Wendler의 531 프로그램은 근력 훈련을 위한 효과적인 루틴으로, 4주..."
1,5/3/1 운동 루틴의 구성 요소와 주간 스케줄에 대해 구체적으로 설명해 주실 수 ...,"5/3/1 운동 루틴은 짐 웨들러가 개발한 웨이트 트레이닝 프로그램으로, 근력 향상..."
2,헬스 입문자로서 531 프로그램의 기본 개념과 구조를 간단히 설명해 주실 수 있나요?,"5/3/1 프로그램은 웨이트 트레이닝 프로그램으로, 근력 향상을 목표로 하는 입문자..."
3,531 루틴의 구성 요소와 훈련 방식을 좀 더 구체적으로 설명해줄 수 있나요?,"531 루틴은 웨이트 트레이닝 프로그램으로 근력 향상을 목표로 하며, 짐 웨들러가 ..."
4,Jim Wendler의 5/3/1 프로그램에서 각 주차별 세트와 반복 구성에 대한 ...,"Jim Wendler의 5/3/1 프로그램은 4주 주기로 구성되어 있으며, 각 주차..."


In [ ]:
df.to_excel(f"/content/drive/MyDrive/05_[공유파일]/Burnfit_dataset/eval_data{step}.xlsx", index=False)